# The Price is Right

Today we'll build another piece of the puzzle: a ScanningAgent that looks for promising deals by subscribing to RSS feeds.

In [9]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from agents.deals import ScrapedDeal, DealSelection

In [10]:
# Initialize and constants

load_dotenv(override=True)
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY', 'your-key-if-not-using-env')
MODEL = 'gpt-4o-mini'
openai = OpenAI()

In [11]:
deals = ScrapedDeal.fetch(show_progress=True)

100%|██████████| 5/5 [01:46<00:00, 21.34s/it]


In [12]:
len(deals)

50

In [13]:
deals[44].describe()

'Title: Armless Desk Chair for $39 + free shipping\nDetails: Apply promo code "AEUS08" to get a total savings of $54. We\'ve pictured it in Dark Gray, but there are several colors available. Buy Now at AliExpress\nFeatures: swivel\nURL: https://www.dealnews.com/Armless-Desk-Chair-for-39-free-shipping/21768956.html?iref=rss-c196'

In [14]:
system_prompt = """You identify and summarize the 5 most detailed deals from a list, by selecting deals that have the most detailed, high quality description and the most clear price.
Respond strictly in JSON with no explanation, using this format. You should provide the price as a number derived from the description. If the price of a deal isn't clear, do not include that deal in your response.
Most important is that you respond with the 5 deals that have the most detailed product description with price. It's not important to mention the terms of the deal; most important is a thorough description of the product.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

{"deals": [
    {
        "product_description": "Your clearly expressed summary of the product in 4-5 sentences. Details of the item are much more important than why it's a good deal. Avoid mentioning discounts and coupons; focus on the item itself. There should be a paragpraph of text for each item you choose.",
        "price": 99.99,
        "url": "the url as provided"
    },
    ...
]}"""

In [15]:
user_prompt = """Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

"""
user_prompt += '\n\n'.join([deal.describe() for deal in deals])

In [16]:
print(user_prompt[:2000])

Respond with the most promising 5 deals from this list, selecting those which have the most detailed, high quality product description and a clear price.
Respond strictly in JSON, and only JSON. You should rephrase the description to be a summary of the product itself, not the terms of the deal.
Remember to respond with a paragraph of text in the product_description field for each of the 5 items that you select.
Be careful with products that are described as "$XXX off" or "reduced by $XXX" - this isn't the actual price of the product. Only respond with products when you are highly confident about the price. 

Deals:

Title: Newegg Shell Shocker Deals: Up to 52% off + free shipping
Details: Save on a large selection of computers, processors, graphics cards, motherboards, external and internal hard drives, SSDs, memory cards, upgrades, and accessories. Some have extra discount coupons on their product pages. Shop Now at Newegg
Features: 
URL: https://www.dealnews.com/Newegg-Shell-Shocker

Below is how we call so we can have structured respose of the `DealSelection` class type

In [17]:
def get_recommendations():
    completion = openai.beta.chat.completions.parse(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
      ],
        response_format=DealSelection
    )
    result = completion.choices[0].message.parsed
    return result

In [18]:
result = get_recommendations()

In [19]:
len(result.deals)

5

In [20]:
result.deals[1]

Deal(product_description='The Vtoman Jump 1500X is a versatile portable power station with an extensive capacity of 828Wh, designed to power multiple devices on the go. It comes with a total of 12 ports, allowing users to efficiently charge various electronics simultaneously. The unit boasts over 3,100 charge cycles, ensuring long-lasting reliability for outdoor adventures or emergency situations. Its robust design and user-friendly interface make it a perfect companion for camping trips, road trips, or as a backup power source at home.', price=324.0, url='https://www.dealnews.com/products/Vtoman/Vtoman-Jump-1500-X-828-Wh-Portable-Power-Station/493915.html?iref=rss-c142')

In [21]:
from agents.scanner_agent import ScannerAgent

In [22]:
agent = ScannerAgent()
result = agent.scan()

In [23]:
result

DealSelection(deals=[Deal(product_description='Experience the robust features of the Ulefone Armor 27 Pro 5G Rugged Smartphone, equipped with a powerful Android 14 operating system. This smartphone boasts a large 6.78-inch FHD+ display with a smooth 120Hz refresh rate, ideal for gaming and media consumption. Its rugged design ensures durability and survivability in extreme conditions, making it perfect for outdoor adventures. With impressive performance and high-quality materials, this smartphone is both versatile and reliable.', price=283.0, url='https://www.dealnews.com/Ulefone-Armor-27-Pro-5-G-Rugged-Smartphone-for-283-free-shipping/21768966.html?iref=rss-c142'), Deal(product_description='The Vtoman Jump 1500X Portable Power Station is designed for those who need portable power on-the-go. With a capacity of 828Wh, it features 12 ports, allowing for multiple devices to be charged simultaneously. This robust power station can withstand over 3,100 cycles, ensuring long-term usability. 